In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [2]:
from unsloth import FastVisionModel, PatchDPOTrainer
from unsloth.trainer import UnslothVisionDataCollator
from trl import DPOConfig, DPOTrainer
from transformers import EarlyStoppingCallback
import torch
import json
from datasets import Dataset
from datasets import load_dataset
from peft import PeftModel

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2026-05-08 10:15:49.287503: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778235349.522782      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778235349.595040      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778235350.185999      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778235350.186049      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778235350.186052      23 computation_placer.cc:177] computation placer alr

🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
# 1. Triệu hồi mô hình B2 (Load lại Adapter cũ)
model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Qwen2-VL-2B-Instruct-bnb-4bit",
    load_in_4bit = True,
)
# Trỏ đúng vào cái não B2 của bạn
model = PeftModel.from_pretrained(
    model, 
    "/kaggle/input/datasets/spixalo/checkpoint-750/checkpoint-750",
    is_trainable=True  # <--- CHÌA KHÓA GIẢI QUYẾT LỖI OOM/TO DEVICE
)

FastVisionModel.for_training(model)

# Kích hoạt PatchDPO cho Vision Model
PatchDPOTrainer()

# =====================================================================
# 2. XỬ LÝ DỮ LIỆU: BƯỚC QUAN TRỌNG NHẤT ĐỂ KHÔNG BỊ LỖI DICT/LIST
# =====================================================================
with open("/kaggle/input/notebooks/spixalo/dl-endterm-rl-dataset/dpo_preference_data_200.json", "r", encoding="utf-8") as f:
    raw_data_list = json.load(f) # Lúc này nó đang là 1 List bình thường

def get_score(item):
    try:
        score_str = item["debug_info"].split(": ")[1]
        return float(score_str)
    except:
        return 1.0 

# Sắp xếp List và cắt lấy 100 mẫu
raw_data_list.sort(key=get_score)
final_100_list = raw_data_list[:100]

# Lưu lại file sạch (tùy chọn)
OUTPUT_FILE = "/kaggle/working/dpo_data_final.json"
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(final_100_list, f, ensure_ascii=False, indent=2)

# ÉP KIỂU SANG HUGGINGFACE DATASET (CHỈ LÀM SAU KHI ĐÃ CẮT 100 MẪU)
final_100_dataset = Dataset.from_list(final_100_list)
# =====================================================================


# 3. Cấu hình "Phòng huấn luyện"
training_args = DPOConfig(
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    num_train_epochs = 6,           
    learning_rate = 5e-6,            
    beta = 0.1,                      
    
    # Cấu hình đánh giá và lưu trữ
    output_dir = "/kaggle/working/qwen2_vl_dpo_checkpoints",
    logging_steps = 5,
    save_strategy = "steps",
    save_steps = 10,                 
    eval_strategy = "steps",
    eval_steps = 10,
    
    load_best_model_at_end = True,
    metric_for_best_model = "eval_loss",
    save_total_limit = 1,
    
    # Tối ưu hóa bộ nhớ
    fp16 = not torch.cuda.is_bf16_supported(),
    bf16 = torch.cuda.is_bf16_supported(),
    optim = "adamw_8bit",
    weight_decay = 0.01,
    remove_unused_columns = False,
)

# 4. Khởi tạo DPOTrainer
trainer = DPOTrainer(
    model = model,
    ref_model = None,                
    args = training_args,
    train_dataset = final_100_dataset,  # <--- Bỏ biến đã ép kiểu vào đây
    eval_dataset = final_100_dataset,   # <--- Bỏ biến đã ép kiểu vào đây
    tokenizer = tokenizer,
    # data_collator = UnslothVisionDataCollator(model, tokenizer),
    callbacks = [EarlyStoppingCallback(early_stopping_patience=2)], 
)

# 5. Bắt đầu uốn nắn hành vi
print("🚀 Đang bắt đầu quá trình Alignment (DPO)...")
trainer.train()

# 6. Lưu bản B3 "xịn xò"
model.save_pretrained("/kaggle/working/qwen2_vl_b3_dpo")
tokenizer.save_pretrained("/kaggle/working/qwen2_vl_b3_dpo")
print("✅ Đã hoàn thành B3! Lưu tại /kaggle/working/qwen2_vl_b3_dpo")

==((====))==  Unsloth 2026.5.2: Fast Qwen2_Vl patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.54G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/572 [00:00<?, ?B/s]

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.


...

Tokenizing eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

🚀 Đang bắt đầu quá trình Alignment (DPO)...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 100 | Num Epochs = 6 | Total steps = 78
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 28,950,528 of 2,237,936,128 (1.29% trained)


Step,Training Loss,Validation Loss,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected,Logits/chosen,Logits/rejected
10,1.054500,1.036061,2.651566,2.953187,0.360000,-0.301621,-34.814884,-34.818520,-1.566698,-1.575970
20,0.910300,0.964071,2.669265,2.860700,0.400000,-0.191435,-34.637894,-35.743401,-1.579564,-1.593618
30,0.924900,0.902746,2.672369,2.770273,0.430000,-0.097904,-34.606857,-36.647667,-1.595357,-1.614037
40,0.738800,0.855134,2.665742,2.686497,0.520000,-0.020755,-34.673126,-37.485424,-1.609625,-1.632146
50,0.756500,0.819847,2.658744,2.614156,0.530000,0.044588,-34.743111,-38.208839,-1.624315,-1.650246
60,0.905700,0.793878,2.649654,2.558075,0.500000,0.091579,-34.834007,-38.769642,-1.632236,-1.660217
70,0.774100,0.781212,2.648426,2.531310,0.510000,0.117116,-34.846283,-39.037292,-1.637544,-1.666510


✅ Đã hoàn thành B3! Lưu tại /kaggle/working/qwen2_vl_b3_dpo
